<a href="https://colab.research.google.com/github/Guldanika/Credit_Card_Fraud_Detection_MlZoomcamp2025_midterm/blob/main/train_model_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile train_model_fraud_detection.py


import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score
import joblib

def main(data_path="creditcard.csv", model_output="model.pkl",
         scaler_output="scaler.pkl", threshold_output="threshold.pkl"):

    # ======================
    # 1. Load Dataset
    # ======================
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"{data_path} not found. Upload it to Colab first.")

    df = pd.read_csv(data_path)
    print(f"Dataset loaded: {df.shape}")

    # ======================
    # 2. Features & Target
    # ======================
    X = df.drop(columns=["Class"])
    y = df["Class"].astype(int)

    # ======================
    # 3. Train/Validation Split
    # ======================
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Train: {X_train.shape}, Validation: {X_val.shape}")

    # ======================
    # 4. Scaling
    # ======================
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # ======================
    # 5. SMOTE Oversampling
    # ======================
    sm = SMOTE(random_state=42)
    X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)
    print(f"After SMOTE, Train: {X_train_res.shape}, Fraud counts: {np.bincount(y_train_res)}")

    # ======================
    # 6. Train Random Forest
    # ======================
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=7,
        class_weight="balanced",
        random_state=42
    )
    rf.fit(X_train_res, y_train_res)
    print("Random Forest trained.")

    # ======================
    # 7. Find Best Threshold by F1
    # ======================
    val_proba = rf.predict_proba(X_val_scaled)[:, 1]
    thresholds = np.linspace(0, 1, 500)
    best_f1 = 0
    best_threshold = 0.5

    for t in thresholds:
        preds = (val_proba >= t).astype(int)
        score = f1_score(y_val, preds)
        if score > best_f1:
            best_f1 = score
            best_threshold = t

    print(f"Best threshold: {best_threshold:.6f}, F1: {best_f1:.6f}")

    # ======================
    # 8. Save Model, Scaler, Threshold
    # ======================
    joblib.dump(rf, model_output)
    joblib.dump(scaler, scaler_output)
    joblib.dump(best_threshold, threshold_output)
    print("Model, scaler, and threshold saved successfully!")

# ======================
# Colab-friendly execution
# ======================
if __name__ == "__main__":
    main()



Overwriting train_model_fraud_detection.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!python train_model_fraud_detection.py \
    --data_path /content/creditcard.csv \
    --model_output model.pkl \
    --scaler_output scaler.pkl \
    --threshold_output threshold.pkl


Dataset loaded: (284807, 31)
Train: (227845, 30), Validation: (56962, 30)
After SMOTE, Train: (454902, 30), Fraud counts: [227451 227451]
Random Forest trained.
Best threshold: 0.879760, F1: 0.797927
Model, scaler, and threshold saved successfully!
